# QLoRA Fine-Tuning: Arabic → German Translation 




In [2]:
# 1. INSTALL REQUIRED LIBRARIES

!pip install -q -U transformers datasets peft accelerate bitsandbytes evaluate sacrebleu sentencepiece


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 90.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 45.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 31.8 MB/s eta 0:00:00


In [3]:
# 2. IMPORTS

import os
import random
import numpy as np
import torch
import evaluate

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType,
    PeftModel,
)


In [4]:
# 3. SEED + GPU CHECK

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print(
        "WARNING: no GPU detected. Go to Settings -> Accelerator -> GPU T4 x2 "
        "in the Kaggle notebook panel, then re-run from the top."
    )


CUDA available: True
GPU: Tesla T4


In [5]:
# 4. CONFIG — tune these for your time budget


# Model
MODEL_NAME = "facebook/mbart-large-50-many-to-many-mmt"

# Dataset (Hugging Face Hub — downloaded automatically, needs Internet ON)
DATASET_NAME = "Helsinki-NLP/multiun"
DATASET_CONFIG = "ar-de"

SOURCE_LANG = "ar"
TARGET_LANG = "de"

# mBART-50 language codes (needed for its tokenizer / generation)
SOURCE_LANG_CODE = "ar_AR"
TARGET_LANG_CODE = "de_DE"

MAX_LEN = 128

# Dataset size (the full multiun ar-de config has ~165k pairs).
# 5,000 trains in well under Kaggle's session time limit on a T4; raise it
# (up to None = full dataset) if you have time budget to spare.
SAMPLE_SIZE = 5000
TEST_SIZE = 0.1

# LoRA / QLoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "out_proj"]  # mBART attention proj names

# Training
NUM_TRAIN_EPOCHS = 2
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 2      # effective batch size = TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS
LEARNING_RATE = 2e-4
EVAL_STEPS = 200
SAVE_STEPS = 200
LOGGING_STEPS = 50

# Output — /kaggle/working persists as the notebook's output
OUTPUT_DIR = "/kaggle/working/ar-de-qlora-finetuned"
ADAPTER_DIR = "/kaggle/working/ar-de-qlora-finetuned/final"

N_SAMPLE_PREDICTIONS = 20


In [6]:
# 5. 4-BIT QUANTIZATION CONFIG

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)


In [16]:
# 6. LOAD TOKENIZER AND QUANTIZED MODEL

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    src_lang=SOURCE_LANG_CODE,
    tgt_lang=TARGET_LANG_CODE,
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

# Make sure generated sequences always start with the German language token
model.generation_config.forced_bos_token_id = tokenizer.convert_tokens_to_ids(TARGET_LANG_CODE)
print("Quantized model loaded")


Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

Quantized model loaded


In [17]:
# 7. PREPARE MODEL FOR LoRA + APPLY LoRA

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()


trainable params: 4,718,592 || all params: 615,598,080 || trainable%: 0.7665


In [18]:
# 8. LOAD + SUBSAMPLE THE DATASET

raw_dataset = load_dataset(DATASET_NAME, DATASET_CONFIG)
full_train = raw_dataset["train"]

print("Full dataset size:", len(full_train))

if SAMPLE_SIZE is not None and len(full_train) > SAMPLE_SIZE:
    full_train = full_train.shuffle(seed=SEED).select(range(SAMPLE_SIZE))

raw_dataset = full_train.train_test_split(test_size=TEST_SIZE, seed=SEED)

print("Train:", len(raw_dataset["train"]))
print("Test :", len(raw_dataset["test"]))


Full dataset size: 165090
Train: 4500
Test : 500


In [19]:
# 9. TOKENIZATION

def preprocess_function(examples):
    inputs = [pair[SOURCE_LANG] for pair in examples["translation"]]
    targets = [pair[TARGET_LANG] for pair in examples["translation"]]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_LEN,
        truncation=True
    )

    labels = tokenizer(
        text_target=targets,
        max_length=MAX_LEN,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenized_dataset = raw_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=raw_dataset["train"].column_names,
)


In [20]:
# 10. DATA COLLATOR

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)


In [21]:
# 11. SACREBLEU METRIC

bleu_metric = evaluate.load("sacrebleu")


def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]
    return preds, labels


def compute_metrics(eval_preds):
    preds, labels = eval_preds

    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result = bleu_metric.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )

    return {"bleu": result["score"]}


In [22]:
# 12. TRAINING ARGUMENTS

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    eval_strategy="steps",
    save_strategy="steps",

    eval_steps=EVAL_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=2,

    learning_rate=LEARNING_RATE,

    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    optim="paged_adamw_8bit",   # paged optimizer reduces memory usage (needs bitsandbytes)

    weight_decay=0.01,
    num_train_epochs=NUM_TRAIN_EPOCHS,

    predict_with_generate=True,
    generation_max_length=MAX_LEN,

    fp16=torch.cuda.is_available(),

    logging_steps=LOGGING_STEPS,

    load_best_model_at_end=True,
    metric_for_best_model="bleu",

    report_to="none",
)

# 13. TRAINER

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


In [23]:
# 14. TRANSLATE FUNCTION

def translate(text, max_length=MAX_LEN):
    tokenizer.src_lang = SOURCE_LANG_CODE

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    generated = model.generate(
        **inputs,
        max_length=max_length,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids(TARGET_LANG_CODE)
    )

    return tokenizer.decode(generated[0], skip_special_tokens=True)


In [24]:
# 15. TRAIN

trainer.train()


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss,Bleu
200,5.318216,1.300204,30.909093
282,5.149847,1.284488,31.601710


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=282, training_loss=5.690612360095302, metrics={'train_runtime': 1021.1765, 'train_samples_per_second': 8.813, 'train_steps_per_second': 0.276, 'total_flos': 2198299760787456.0, 'train_loss': 5.690612360095302, 'epoch': 2.0})

In [25]:
# 16. SAVE LoRA ADAPTER


model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print("LoRA adapter saved to:", ADAPTER_DIR)


LoRA adapter saved to: /kaggle/working/ar-de-qlora-finetuned/final


In [26]:
# 17. EVALUATE ON TEST SET

print("\n===== Evaluation on test set =====")
eval_results = trainer.evaluate()

for key, value in eval_results.items():
    print(f"{key}: {value:.4f}" if isinstance(value, float) else f"{key}: {value}")



===== Evaluation on test set =====


Training Loss,Validation Loss,Step,Bleu
5.149847,1.284488,282,31.601710


eval_loss: 1.2845
eval_bleu: 31.6017


In [27]:
# 18. SAMPLE PREDICTIONS

print("\n===== Sample predictions =====")

test_raw = raw_dataset["test"]
num_samples = min(N_SAMPLE_PREDICTIONS, len(test_raw))
sample_indices = random.sample(range(len(test_raw)), num_samples)

for idx in sample_indices:
    arabic_sentence = test_raw[idx]["translation"][SOURCE_LANG]
    actual_german = test_raw[idx]["translation"][TARGET_LANG]
    predicted_german = translate(arabic_sentence)

    print(f"Arabic    : {arabic_sentence}")
    print(f"Actual DE : {actual_german}")
    print(f"Predicted : {predicted_german}")
    print()



===== Sample predictions =====
Arabic    : وإذ تشير إلى أحكام الإعلان العالمي لحقوق الإنسان()، فضلا عن المادة 12 من العهد الدولي الخاص بالحقوق المدنية والسياسية()،
Actual DE : unter Hinweis auf die Bestimmungen der Allgemeinen Erklärung der Menschenrechte und auf Artikel 12 des Internationalen Paktes über bürgerliche und politische Rechte,
Predicted : unter Hinweis auf die Resolutionen der Universal Declaration of Human Rights sowie den Artikel 12 des Internationalen Übereinkommens über bürgerliche und politische Rechte,

Arabic    : 22 - يشجع اللجنة على كفالة وجود إجراءات عادلة وواضحة يتم بموجبها إدراج الأفراد والكيانات في قائمة اللجنة ورفع أسمائهم منها، فضلا عن منح استثناءات لأسباب إنسانية؛
Actual DE : 22. ermutigt den Ausschuss, dafür Sorge zu tragen, dass faire und klare Verfahren vorhanden sind, die die Aufnahme von Personen und Einrichtungen in die Liste des Ausschusses und ihre Streichung von dieser Liste sowie die Gewährung von Ausnahmen aus humanitären Gründen regeln;
Predict

In [28]:
# 19. CUSTOM SENTENCE

custom_sentence = "الباب لن يفتح."

print("===== Custom sentence =====")
print("Arabic:", custom_sentence)
print("German:", translate(custom_sentence))


===== Custom sentence =====
Arabic: الباب لن يفتح.
German: Die Tür wird nicht geöffnet.
